## SDK 0.2.1 설치 / Install SDK 0.2.1
첫 셀은 SDK 0.2.1이 없을 때만 고정 wheel을 설치합니다(네트워크 필요). 설치된 경우 건너뜁니다. 결과 전송은 별도 선택이며 기본 False입니다.
The first cell installs the pinned wheel only if SDK 0.2.1 is absent (network required). If 0.2.1 is already installed, installation is skipped. Publishing results is a separate choice, default False.


In [ ]:
# Install only when this SDK version is absent. No result publication.
import importlib
import subprocess
import sys

try:
    import luna_repro
    current_sdk = getattr(luna_repro, "__version__", None)
except ModuleNotFoundError as error:
    if error.name != "luna_repro":
        raise
    current_sdk = None

if current_sdk != "0.2.1":
    subprocess.check_call([sys.executable, "-m", "pip", "install",
        "https://github.com/lunalab-ai/luna-repro-sdk/releases/download/v0.2.1/luna_repro-0.2.1-py3-none-any.whl#sha256=86a1a5fbe26c203241676cd97e0bb37a49d00d0a6b82cb951ba5f1c0bae40efa"])
    if current_sdk is not None:
        raise RuntimeError("SDK updated. Restart the runtime/kernel, then Run all again. SDK 업데이트 후 런타임을 재시작하고 다시 실행하세요.")
    importlib.invalidate_caches()
    import luna_repro
assert luna_repro.__version__ == "0.2.1"
print("LUNA Repro SDK", luna_repro.__version__)


# 재현성 첫걸음 / Reproducibility walkthrough

설치 → 합성 계산 → 선택 결과 저장 → 선택적 전송 순서입니다. 기본 `PUBLISH = False`이므로 SDK 설치 이후 계산·저장 셀은 네트워크·GPU·AI를 사용하지 않습니다. 실제 연구 성능이 아닌 6개 교육용 합성 값입니다.
Install → compute synthetic values → save selected results → optionally publish. With the default `PUBLISH = False`, after SDK installation, calculation/save cells make no network, GPU or AI call. These six teaching values are not research performance.

## 1. SDK 설치 / Install the SDK
앞의 첫 설치 셀을 실행하세요. SDK 0.2.1이 없으면 버전·해시가 고정된 공개 wheel을 설치합니다(네트워크 필요). 이미 설치되어 있으면 건너뜁니다. 기존 ZIP을 직접 설치하는 방법은 대체 경로이며 필수 준비가 아닙니다.
Run the first installation cell above. If SDK 0.2.1 is absent, it installs the public wheel pinned by version and hash (network required); an existing 0.2.1 installation is skipped. Installing a downloaded ZIP manually is an alternative, not a prerequisite.


In [ ]:
import math
import os
import struct
import uuid
import zlib
from datetime import datetime, timezone
from pathlib import Path
import luna_repro as repro

OFFLINE_PROJECT = "12345678-1234-5678-1234-567812345678"
PROJECT_ID = OFFLINE_PROJECT  # 오프라인 연습용 / Offline practice only
PUBLISH = False  # 실제 전송을 원할 때만 True / True only for explicit publication

# Saved-notebook source capture is explicit; it does not identify the running cell automatically.
INCLUDE_SOURCE = False
SOURCE_NOTEBOOK = Path("walkthrough.ipynb")  # Saved copy uploaded to this runtime
SOURCE_PROVIDER = "jupyter"  # "colab" only for a saved Colab notebook
SOURCE_CELL_ID = "calculation-and-pages"  # Verify actual id or metadata.id in that saved copy
SOURCE_NOTEBOOK_URL = None  # Clean Colab notebook URL, when using provider="colab"


## 2. 계산 코드를 확인합니다 / Inspect the calculation
`reference = [0,1,2,3,4,5]`, `measured = reference + 0.5`, `corrected = measured - 0.4`입니다. 다음 셀은 함수만 정의합니다. 표준 라이브러리로 그래프 PNG를 만들어 추가 설치가 필요 없습니다.
The following cell defines the calculation, a small PNG chart and page assembly. It uses only the standard library and the installed SDK. No files or network calls are made when functions are defined.

브라우저의 완성 예시(잡음 신호+이동평균)와는 다른 합성 예제입니다. 이 노트북은 MAE/RMSE 0.5 → 약 0.1을 예상합니다.
This is a different synthetic example from the browser's noisy-signal/moving-average preview. This notebook expects MAE/RMSE 0.5 → approximately 0.1.


In [ ]:
def calculate():
    """Six known targets, a +0.5 offset, and an explicitly chosen correction."""
    reference = [float(index) for index in range(6)]
    measured = [value + 0.5 for value in reference]
    corrected = [value - 0.4 for value in measured]
    return reference, measured, corrected


def summarize(reference, prediction):
    errors = [actual - expected for actual, expected in zip(prediction, reference)]
    return {
        "MAE": sum(abs(error) for error in errors) / len(errors),
        "RMSE": math.sqrt(sum(error * error for error in errors) / len(errors)),
        "samples": len(errors),
    }


def chart_png(reference, prediction):
    """Draw the selected six points as a small RGB PNG without extra packages."""
    width, height = 640, 300
    pixels = bytearray([250, 251, 253] * width * height)

    def point(x, y, color):
        if 0 <= x < width and 0 <= y < height:
            offset = 3 * (y * width + x)
            pixels[offset:offset + 3] = bytes(color)

    def line(x0, y0, x1, y1, color):
        steps = max(abs(x1 - x0), abs(y1 - y0), 1)
        for step in range(steps + 1):
            x = round(x0 + (x1 - x0) * step / steps)
            y = round(y0 + (y1 - y0) * step / steps)
            point(x, y, color)
            point(x, y + 1, color)

    for y in range(30, height - 20, 40):
        line(30, y, width - 25, y, (220, 225, 232))
    for values, color in ((reference, (35, 80, 170)), (prediction, (210, 95, 25))):
        coords = [(30 + index * 115, round(height - 30 - value * 40))
                  for index, value in enumerate(values)]
        for start, end in zip(coords, coords[1:]):
            line(*start, *end, color)
        for x, y in coords:
            for dx in range(-3, 4):
                for dy in range(-3, 4):
                    point(x + dx, y + dy, color)

    def chunk(kind, content):
        return (struct.pack(">I", len(content)) + kind + content
                + struct.pack(">I", zlib.crc32(kind + content) & 0xffffffff))

    raw = b"".join(b"\x00" + pixels[row * width * 3:(row + 1) * width * 3]
                   for row in range(height))
    return (b"\x89PNG\r\n\x1a\n"
            + chunk(b"IHDR", struct.pack(">IIBBBBB", width, height, 8, 2, 0, 0, 0))
            + chunk(b"IDAT", zlib.compress(raw)) + chunk(b"IEND", b""))


def build_pages(project=OFFLINE_PROJECT, *, outbox=None):
    """Return two complete Page objects; do not save or publish them."""
    reference, measured, corrected = calculate()
    pages = []
    for label, prediction in (("baseline", measured), ("corrected", corrected)):
        page = repro.page(
            project,
            "따라하기 데모 / Walkthrough demo · " + label,
            description=("6개 합성 값으로 결과 등록을 익히는 CPU 예제입니다. "
                         "Six synthetic values teach result registration; not research performance."),
            metadata={"tags": ["synthetic", "walkthrough", "CPU", label]},
            evidence={"origin": "synthetic", "execution_status": "completed",
                      "evaluation_status": "completed"},
            source={"file": "walkthrough.ipynb", "cell_id": "calculation-and-pages"},
            outbox=outbox,
        )
        page.section("conditions", "계산 조건 / Conditions").text(
            "reference = [0, 1, 2, 3, 4, 5]; measured = reference + 0.5; "
            "corrected = measured - 0.4. CPU, Python standard library, no random input. "
            "MAE = mean(abs(prediction - reference)); RMSE = sqrt(mean(error ** 2))."
        )
        page.section("metrics", "직접 계산한 지표 / Computed metrics").metrics(
            summarize(reference, prediction)
        )
        page.section("plot", "기준과 결과 / Reference and result").image(
            chart_png(reference, prediction),
            "파랑: 기준, 주황: 선택 결과 / Blue: reference; orange: selected result",
            name=label + ".png",
        )
        page.section("samples", "6개 값 확인 / Inspect all six values").table(
            ["index", "reference", "prediction", "error"],
            [[index, expected, actual, actual - expected]
             for index, (expected, actual) in enumerate(zip(reference, prediction))],
        )
        page.section("interpretation", "해석과 한계 / Interpretation and limits").text(
            "baseline의 MAE/RMSE는 0.5, corrected는 약 0.1입니다. "
            "알려진 오프셋을 보정한 교육용 계산이며 새 데이터 일반화·독립 재현 검증이 아닙니다. "
            "Baseline MAE/RMSE = 0.5; corrected is approximately 0.1. "
            "This corrects a known synthetic offset; it does not establish generalization "
            "or independent reproduction."
        )
        pages.append(page)
    return pages



## 3. 결과를 계산하고 로컬 저장 / Compute and save locally
연결 없이 실행할 수 있습니다. 전송하려면 먼저 `PROJECT_ID`를 결과 연결 화면의 **실제 프로젝트 UUID**로 바꾸고 이 셀을 다시 실행하세요. 프로젝트 UUID는 비밀 키와 다릅니다. 매번 새 실행 폴더를 만들어 이전 파일을 보존합니다.
You can run this offline. Before publication, set `PROJECT_ID` above to your **real project UUID** from Connect results and rerun this cell. The project UUID is not the connection key. Each run gets a new folder, preserving earlier files.

선택 코드 보존: 현재 노트북을 파일로 저장해 실행 환경에 올린 뒤 setup의 SOURCE_NOTEBOOK 경로와 실제 셀 ID를 확인하고 INCLUDE_SOURCE=True로 바꾸세요. 기본값 False는 파일을 읽지 않습니다. Jupyter의 셀 id와 Colab metadata.id는 서로 다를 수 있습니다. 저장한 사본은 실행 당시 코드와 같다고 단정할 수 없어 참고 코드로 표시됩니다.
Optional code preservation: save a notebook copy and upload it to the runtime. Verify SOURCE_NOTEBOOK and the actual saved cell identity, then set INCLUDE_SOURCE=True. False does not read this file. Jupyter id and Colab metadata.id can differ. A saved copy is labeled reference code; it does not establish identity with an earlier execution.


In [ ]:
PROJECT_ID = str(uuid.UUID(PROJECT_ID))
run_folder = Path(".luna-repro/walkthrough-notebook") / (
    datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ") + "-" + uuid.uuid4().hex[:8]
)
pages = build_pages(PROJECT_ID, outbox=run_folder / "outbox")
if INCLUDE_SOURCE:
    # Only the selected saved code cell; no outputs, runtime globals or directory upload.
    selected = repro.source.notebook(
        SOURCE_NOTEBOOK, SOURCE_CELL_ID, root=SOURCE_NOTEBOOK.resolve().parent,
        provider=SOURCE_PROVIDER, notebook_url=SOURCE_NOTEBOOK_URL,
        snapshot=True, role="reference",
    )
    for page in pages:
        page.source = selected
        for section in page.bundle()["sections"]:
            page.section(section["key"], source=selected)
for label, page in zip(("baseline", "corrected"), pages):
    saved_path = page.save(run_folder / (label + ".json"))
    print(label, page.bundle()["sections"][1]["blocks"][0]["values"])
    print("Saved:", saved_path)
print("Offline results saved. No publication has happened.")


## 4. 선택한 경우에만 실제 등록 / Publish only when you choose
먼저 프로젝트 UUID를 바꾸고 3번 셀을 다시 실행합니다. 그다음 아래 `PUBLISH`를 `True`로 바꾸고 이 셀만 실행합니다. 키는 숨김 입력창으로 받으며 소스·출력에 적지 않습니다. 입력 취소는 전송하지 않습니다. `published`만 플랫폼 등록 성공이고 `pending`은 로컬 대기입니다.
First replace the project UUID and rerun step 3. Then set `PUBLISH` below to `True` and run this cell. Enter your key through the hidden prompt, never source or output. Only `published` means accepted by the platform; `pending` means preserved locally for explicit retry.


In [ ]:
PUBLISH = False  # 원할 때만 True로 바꾸세요 / Change only when ready
if PUBLISH:
    if PROJECT_ID == OFFLINE_PROJECT:
        raise ValueError("Set your real PROJECT_ID and rerun the compute/save cell first.")
    import getpass
    os.environ["LUNA_REPRO_ENDPOINT"] = "https://lunalab-ai.com"
    os.environ["LUNA_REPRO_TOKEN"] = getpass.getpass("LUNA project key (hidden): ")
    for page in pages:
        receipt = page.publish()
        if receipt["status"] == "published":
            print("Published:", receipt["url"], "revision", receipt["revision"])
        else:
            print("Pending:", receipt["code"], receipt["message"])
            print("Preserved:", receipt["outbox_path"])
else:
    print("Publication is off. Set the real project first, then explicitly enable this cell.")


## 5. 실패 후 명시적 재시도 / Explicit retry after a failed transfer
키·주소·권한을 수정한 뒤 아래 `RETRY = True`로 바꾸고 실행합니다. 새로운 페이지를 만들지 않고 같은 outbox의 미전송 요청만 다시 보냅니다. rate_limited이면 반환된 retry_after만큼 기다린 뒤 직접 실행합니다.
After fixing the connection, set `RETRY = True` below. It retries pending requests from this run's outbox without creating a new Page. For rate_limited, wait for retry_after before explicitly trying again.


In [ ]:
RETRY = False
if RETRY:
    for receipt in repro.sync(run_folder / "outbox"):
        print(receipt["status"], receipt.get("url") or receipt["code"])
else:
    print("Retry is off.")


## 6. 플랫폼에서 확인 / Continue on the platform
성공 URL을 열고 지표·그래프·6행 표·코드 출처를 확인합니다. 두 결과에서 섹션을 선택해 하나의 보고서를 만드세요. 원본의 고정 버전과 나만 보기 설정을 확인합니다. 프로젝트 공유·Notion 전달·AI 상담은 각각 사용자가 선택하는 별도 행동입니다. 결과 등록 성공은 독립 재현 검증을 뜻하지 않습니다.
Open each successful URL. Inspect metrics, chart, all six table rows and provenance. Select sections from both pages and compose one report. Check pinned versions and private visibility. Project sharing, Notion delivery and AI consultation are separate choices. Successful receipt is not independent reproduction.

자세한 매뉴얼 / Full manual: https://lunalab-ai.com/workspace/repro/guide/
